In [3]:
import pandas as pd
import glob
import os
import numpy as np

In [2]:
WaterLevel_v1_cols = ["sys_loc_code", 'measurement_date', 'equipment_code', 'historical_reference_elev', 'water_level_depth', 'water_level_elev', 'corrected_depth', 'corrected_elev',
               'measured_depth_of_well', 'depth_unit', 'technician', 'dry_indicator_yn', 'measurement_method', 'batch_number', 'dip_or_elevation', 'remark', 'lnapl_cas_rn',
               'lnapl_depth', 'dnapl_cas_rn', 'dnapl_depth', 'task_code', 'approval_code', 'custom_field_1', 'custom_field_2', 'custom_field_3', 'custom_field_4', 'TRC_Project_Number',
               'reportable_yn', 'fld_qualifier', 'fld_qualifier_note']

In [3]:
# concatenate all excel files in the directory into a single dataframe, then export to csv
base_path = r"C:\Users\ageglio\OneDrive - TRC\Documents\Mosaic Hutchinson\Historical data"
files = glob.glob(os.path.join(base_path,"*", '*.xlsx'))
mhwl_df = pd.DataFrame(columns=WaterLevel_v1_cols)
column_rename = {"well location":"sys_loc_code", 
                 "date": "measurement_date", 
                 "reference elevation (feet, msl)":"historical_reference_elev", 
                 "depth to water (feet)":"water_level_depth", 
                 "depth to bottom (feet)":"measured_depth_of_well"}
for f in files:
    df = pd.read_excel(f)
    # clean headers by removing leading/trailing whitespace, newlines, and converting to lowercase
    df.columns = df.columns.str.strip().str.replace('\n', ' ').str.lower()
    # confert date to MM/DD/YYYY format
    df['date'] = pd.to_datetime(df['date']).dt.strftime('%m/%d/%Y')
    # rename columns to match EDD template
    df = df.rename(columns=column_rename)
    mhwl_df = pd.concat([mhwl_df, df], ignore_index=True)

In [4]:
# convert "CNL" in any column to "Could Not Locate"
mhwl_df = mhwl_df.replace("CNL", "Could Not Locate")
# convert "DNM" in any column to "Did Not Measure"
mhwl_df = mhwl_df.replace("DNM", "Did Not Measure")
# convert "NM" in any column to "Not Measured"
mhwl_df = mhwl_df.replace("NM", "Not Measured")

# find any rows where water level depth indicates dry well (e.g. "dry", "dry well", "no water", etc.) and move that text to the remark column, and make water level depth "NM" for those rows
dry_idx = mhwl_df[mhwl_df['water_level_depth'].str.contains(r"dry|no water|bottom\b", case=False, na=False)].index
mhwl_df.loc[dry_idx, 'dry_indicator_yn'] = "Y"
mhwl_df.loc[~mhwl_df.index.isin(dry_idx), 'dry_indicator_yn'] = "N"

# populate the measurement method column based on whether the sys_loc_code contains "RW" 
# (assume "RW" indicates a recovery well that was measured with a laser, and all other wells were measured with a transducer)
recovery_well_idx = mhwl_df[mhwl_df['sys_loc_code'].str.contains("RW", case=False, na=False)].index
mhwl_df.loc[recovery_well_idx, "measurement_method"] = "transducer"
mhwl_df.loc[~mhwl_df.index.isin(recovery_well_idx), "measurement_method"] = "tape"

mhwl_df['reportable_yn'] = "Y"
mhwl_df['dip_or_elevation'] = "dip"
mhwl_df['depth_unit'] = "ft"

# find rows where the water level depth has any alph character comments (e.g. "bgs", "estimated", etc.)
rmk_idx = mhwl_df[mhwl_df['water_level_depth'].str.contains(r"[a-zA-Z]", case=False, na=False)].index

mhwl_df2 = mhwl_df.copy()
mhwl_df2.loc[rmk_idx, 'remark'] = mhwl_df.loc[rmk_idx, 'water_level_depth']
mhwl_df2.loc[rmk_idx, 'water_level_depth'] = ""

# change measurement type in remark idx to "NM" since these are not actual measurements
mhwl_df2.loc[rmk_idx, 'measurement_method'] = "NM"

mhwl_df2.loc[rmk_idx, 'dry_indicator_yn'] = ""

# clean up ref elev column by making them nan if they contain any alphabetic characters (e.g. "msl", "estimated", etc.)
hre_rmk = mhwl_df2[mhwl_df2['historical_reference_elev'].str.contains(r"[a-zA-Z]", case=False, na=False)].index
mhwl_df2.loc[hre_rmk, 'historical_reference_elev'] = ""

# # clean up measured depth of well column by making them nan if they contain any alphabetic characters (e.g. "roots", "obstructed", etc.)
mdw_rmk = mhwl_df2[mhwl_df2['measured_depth_of_well'].str.contains(r"[a-zA-Z>]", case=False, na=False)].index
mhwl_df2.loc[mdw_rmk, 'measured_depth_of_well'] = ""
mhwl_df2.to_csv(os.path.join(base_path, "Mosaic_Hutchinson_WaterLevel_v1.csv"), index=False, header=False)

In [10]:
mhwl_df2[mhwl_df2['sys_loc_code'] == "MW-24S"]

,sys_loc_code,measurement_date,equipment_code,historical_reference_elev,water_level_depth,water_level_elev,corrected_depth,corrected_elev,measured_depth_of_well,depth_unit,...,task_code,approval_code,custom_field_1,custom_field_2,custom_field_3,custom_field_4,TRC_Project_Number,reportable_yn,fld_qualifier,fld_qualifier_note
69,MW-24S,01/01/2007,NaN,1511.33,9.86,NaN,NaN,NaN,19.14,ft,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Y,NaN,NaN
151,MW-24S,04/01/2007,NaN,1511.33,6.80,NaN,NaN,NaN,19.14,ft,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Y,NaN,NaN
200,MW-24S,07/01/2007,NaN,1511.33,7.45,NaN,NaN,NaN,19.14,ft,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Y,NaN,NaN
293,MW-24S,10/01/2007,NaN,1511.33,8.02,NaN,NaN,NaN,19.14,ft,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Y,NaN,NaN
383,MW-24S,03/10/2008,NaN,1511.33,9.38,NaN,NaN,NaN,NaN,ft,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Y,NaN,NaN
473,MW-24S,06/16/2008,NaN,1511.33,6.60,NaN,NaN,NaN,NaN,ft,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Y,NaN,NaN
564,MW-24S,09/15/2008,NaN,1511.33,6.96,NaN,NaN,NaN,NaN,ft,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Y,NaN,NaN
655,MW-24S,11/17/2008,NaN,1511.33,,NaN,NaN,NaN,NaN,ft,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Y,NaN,NaN
750,MW-24S,02/09/2009,NaN,1511.33,7.64,NaN,NaN,NaN,NaN,ft,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Y,NaN,NaN
845,MW-24S,04/27/2009,NaN,1511.33,7.52,NaN,NaN,NaN,NaN,ft,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Y,NaN,NaN


In [7]:
ref_first = (
    mhwl_df2.assign(
        historical_reference_elev=pd.to_numeric(mhwl_df2['historical_reference_elev'], errors='coerce'),
        measurement_date=pd.to_datetime(mhwl_df2['measurement_date'], errors='coerce')
    )
    .dropna(subset=['historical_reference_elev'])
    .groupby(['sys_loc_code', 'historical_reference_elev'], as_index=False)['measurement_date']
    .min()
    .rename(columns={'measurement_date': 'first_observed'})
)
well_v1_cols = ["sys_loc_code", "well_id", "well_description", "well_owner", "well_purpose", "well_status", "top_casing_elev", "datum_value", "datum_unit", "datum_desc", "step_or_linear", "datum_start_date", "datum_collection_method_code", "depth_of_well", "depth_unit", "depth_to_bedrock", "depth_measure_method", "stickup_height", "stickup_unit", "sump_length", "sump_unit", "installation_date", "construct_start_date", "construct_complete_date", "construct_contractor", "pump_type", "pump_capacity", "pump_unit", "pump_yield", "pump_yield_method", "weep_hole", "head_configuration", "access_port_yn", "casing_joint_type", "perforator_used", "intake_depth", "disinfected_yn", "historical_reference_elev", "geologic_unit_code", "remark"]
well_v1 = pd.DataFrame(columns=well_v1_cols)
well_v1 = well_v1.assign(**{"sys_loc_code": ref_first["sys_loc_code"], "datum_value": ref_first["historical_reference_elev"], "datum_start_date": ref_first["first_observed"]})
well_v1['step_or_linear'] = "step"
well_v1['datum_unit'] = "ft"
well_v1.to_csv(os.path.join(base_path, "Mosaic_Hutchinson_Wells_v1.csv"), index=False, header=False)

In [ ]:
loc_v1_columns = ['data_provider',	'sys_loc_code',	'x_coord',	'y_coord', 'surf_elev', 'elev_unit',	
                  'coord_type_code',	
                  'observation_date',	
                  'coord_identifier',	
                  'horz_collect_method_code',	
                  'horz_accuracy_value',	
                  'horz_accuracy_unit',	
                  'horz_datum_code',	
                  'elev_collect_method_code',	
                  'elev_accuracy_value',	
                  'elev_accuracy_unit',	
                  'elev_datum_code',	
                  'source_scale',	
                  'subcontractor_name_code',	
                  'verification_code',	
                  'reference_point',	
                  'geometric_type_code',	
                  'rank',	
                  'loc_name',	
                  'loc_desc',	
                  'loc_type',	
                  'loc_purpose',	
                  'subfacility_code',	
                  'within_facility_yn',	
                  'loc_county_code',	
                  'loc_district_code',	
                  'loc_state_code',	
                  'loc_major_basin',	
                  'loc_minor_basin',	
                  'remark',	
                  'total_depth',	
                  'depth_unit',	
                  'geologist',	
                  'inspector',	
                  'bore_id',	
                  'loc_type_2',	
                  'log_date',	
                  'stream_code',	
                  'stream_mile',	
                  'remark_2',	
                  'custom_field_1',	
                  'custom_field_2',	
                  'custom_field_3',	
                  'custom_field_4',	
                  'custom_field_5',	
                  'parent_loc_code',	
                  'land_use'
]

In [ ]:
new_sys_loc_codes = [
]
loc_df = pd.DataFrame(columns=loc_v1_columns)
loc_df['sys_loc_code'] = new_sys_loc_codes
# loc_df.to_csv(os.path.join(base_path, "Mosaic_Hutchinson_loc.csv"), index=False, header=False)

In [ ]:
# import matplotlib.pyplot as plt
# # plot line plot of water level depth over time for each well
# mw15 = mhwl_df2[mhwl_df2['sys_loc_code'].str.contains(r"CMW-15", case=False, na=False)]
# for well in mw15['sys_loc_code'].unique():
#     well_data = mw15[mw15['sys_loc_code'] == well]
#     # convert blanks to np.nan and convert water level depth to float for plotting
#     well_data['water_level_depth'] = pd.to_numeric(well_data['water_level_depth'], errors='coerce')
#     well_data['historical_reference_elev'] = pd.to_numeric(well_data['historical_reference_elev'], errors='coerce')
    
#     # convert the date strings to datetime objects for better plotting
#     well_data['measurement_date'] = pd.to_datetime(well_data['measurement_date'])

#     dates = well_data['measurement_date']
#     depths = well_data['water_level_depth']
#     elev = well_data['historical_reference_elev']
#     water_level_elev = elev - depths
#     plt.plot(dates, water_level_elev, label=well)

# plt.hlines(elev.mean(), xmin=dates.min(), xmax=dates.max(), colors='gray', linestyles='dashed', label=f"Ref Elev")

# plt.xlabel('Measurement Date')
# plt.ylabel('Water Level Elevation')
# plt.title('Water Level Elevation Over Time for Each Well')
# plt.legend()
# plt.show()